# 01 — Data Exploration

Explore the Anthropic HH-RLHF dataset:
- Distribution of chosen vs rejected response lengths
- Typical conversation structures
- Token count statistics after GPT-2 tokenisation
- What the preference signal looks like at the surface level

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_from_disk
from transformers import GPT2Tokenizer

In [ ]:
# Run data/prepare_hh_rlhf.py first
dataset = load_from_disk('../data/processed')
train = dataset['train']
print(f"Train: {len(train):,}  Val: {len(dataset['val']):,}  Test: {len(dataset['test']):,}")

In [ ]:
# Token length distributions
chosen_lens  = [len(x) for x in train['chosen_input_ids']]
rejected_lens = [len(x) for x in train['rejected_input_ids']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(chosen_lens,  bins=50, alpha=0.7, label='chosen',   color='steelblue')
axes[0].hist(rejected_lens, bins=50, alpha=0.7, label='rejected', color='salmon')
axes[0].set_xlabel('Token length')
axes[0].set_ylabel('Count')
axes[0].set_title('Chosen vs Rejected token lengths')
axes[0].legend()

length_diffs = np.array(chosen_lens) - np.array(rejected_lens)
axes[1].hist(length_diffs, bins=50, color='mediumpurple', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('chosen_len − rejected_len (tokens)')
axes[1].set_title('Length difference (chosen − rejected)')

plt.tight_layout()
plt.savefig('../results/figures/01_token_length_distribution.png', dpi=150)
plt.show()

print(f"Chosen    mean={np.mean(chosen_lens):.0f}  median={np.median(chosen_lens):.0f}  max={max(chosen_lens)}")
print(f"Rejected  mean={np.mean(rejected_lens):.0f}  median={np.median(rejected_lens):.0f}  max={max(rejected_lens)}")
print(f"\nChosen is longer in {(np.array(length_diffs) > 0).mean():.1%} of pairs")

In [ ]:
# Inspect a few examples
import random
random.seed(42)
indices = random.sample(range(len(train)), 3)

for i in indices:
    row = train[i]
    print('=' * 70)
    print('PROMPT:')
    print(row['prompt'][-300:])  # last 300 chars of prompt
    print('\nCHOSEN RESPONSE:')
    print(row['chosen_response'][:300])
    print('\nREJECTED RESPONSE:')
    print(row['rejected_response'][:300])
    print()